# 07 — Final Submission

**What this notebook is for.** We've selected our final model: an **ensemble of
CatBoost + the lab-focused GRU**, which reached OOF AUROC ~0.812, above either
single model. This notebook produces the actual deliverable — predictions on the
project's `test.csv` — by:

1. Retraining **both base models on ALL labeled data** (train + valid combined).
2. Building features for the **test** admissions exactly as for training.
3. Predicting test with each model and combining by **rank average** (scale-safe,
   and it matched the weighted blend on OOF).
4. Writing the submission file.

**Key point on test.** Until now we never touched `test`. Here we finally use it —
but only to *predict*, never to fit or select anything. All model choices were
made on train + valid. Its true AUROC is not visible to us; our best estimate of
test performance is the OOF ~0.812.

**Why rank average for the final blend.** The OOF results showed rank average
(0.8118) and weighted (0.8121) were statistically identical. Rank average is
scale-invariant — it doesn't care that a tree and a neural net put probabilities
on different scales — so it's the robust choice when combining fresh predictions
on new data.

**How to read this notebook.** Each code cell has a plain-language note.

## 0. Setup and GPU

In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


**How to read this:** confirms the GPU for the GRU. CPU works too, just slower.

## 1. Load all labeled data and the test set

We combine train + valid into one labeled set (all the data we're allowed to
learn from), and load `test.csv` (predict-only). We also load the raw sequences
for the GRU.

In [2]:
DATA_DIR = "data"
LABEL_COL = "readmitted_within_30days"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

def to_binary(s):
    return s.map(lambda v: 1 if v in {1, "1", True, "True"} else 0).astype(int)
train_df["y"] = to_binary(train_df[LABEL_COL])
valid_df["y"] = to_binary(valid_df[LABEL_COL])

labeled = pd.concat([train_df, valid_df], ignore_index=True).drop_duplicates("id").reset_index(drop=True)
test = test_df.drop_duplicates("id").reset_index(drop=True)

with open(os.path.join(DATA_DIR, "ehr_preprocessed_seq_by_day_cat_embedding.pkl"), "rb") as f:
    feat_dict = pickle.load(f)["feat_dict"]

train_ids = labeled["id"].tolist()
test_ids  = test["id"].tolist()
y_train = labeled["y"].values.astype(np.float32)

print("labeled admissions:", len(train_ids), "| positive rate:", round(y_train.mean(), 4))
print("test admissions to predict:", len(test_ids))
# confirm every test id has feature data
missing = [i for i in test_ids if i not in feat_dict]
print("test ids missing feature data:", len(missing))

labeled admissions: 11022 | positive rate: 0.1751
test admissions to predict: 2741
test ids missing feature data: 0


**How to read this:** all labeled data is pooled for training; test is separate
and predict-only. The missing-data check should be 0 — every test admission needs
its feature array. **Remember:** we build test features with the *same* functions
as training, so the two are directly comparable.

## 2. Feature builders (identical for train and test)

The exact functions from the feature/ensemble notebooks: the flat table for
CatBoost, and the lab-window + static vector for the GRU. Using one set of
functions for both train and test guarantees the test features are built
identically — a common place for bugs if train and test are processed
differently.

In [3]:
DEMO, ICD, LAB, MED = slice(0,3), slice(3,94), slice(94,130), slice(130,171)
N_DAYS, N_LAB = 7, 36

# --- flat table (CatBoost) ---
def build_flat(ids):
    rows = []
    for i in ids:
        m = feat_dict[i]
        demo = m[0, DEMO]
        icd_flags = (m[:, ICD].max(0) > 0).astype(np.float32)
        med_flags = (m[:, MED].max(0) > 0).astype(np.float32)
        med_int   = m[:, MED].sum(0) / m.shape[0]
        lab_last  = m[-1, LAB].astype(np.float32)
        lab_ever  = (m[:, LAB].max(0) > 0).astype(np.float32)
        lab_frac  = m[:, LAB].mean(0)
        seq_len   = np.array([m.shape[0]], dtype=np.float32)
        rows.append(np.concatenate([
            demo, [icd_flags.sum()], icd_flags,
            [med_flags.sum()], med_flags, med_int,
            lab_last, lab_ever, lab_frac, seq_len]))
    cols = (["age","gender","ethnicity","icd_count"] + [f"icd_flag__{j}" for j in range(91)]
            + ["med_count"] + [f"med_flag__{j}" for j in range(41)] + [f"med_intensity__{j}" for j in range(41)]
            + [f"lab_last__{j}" for j in range(36)] + [f"lab_flag__{j}" for j in range(36)]
            + [f"lab_frac__{j}" for j in range(36)] + ["sequence_length"])
    df = pd.DataFrame(np.vstack(rows), columns=cols)
    # np.vstack makes everything float; CatBoost needs categoricals as int/string
    for c in ["gender", "ethnicity"]:
        df[c] = df[c].round().astype(int)
    return df

# --- GRU inputs ---
def build_lab_static(ids):
    Xlab = np.zeros((len(ids), N_DAYS, N_LAB), dtype=np.float32)
    stat = []
    for k, i in enumerate(ids):
        m = feat_dict[i]
        seq = m[-N_DAYS:, LAB]
        if seq.shape[0] < N_DAYS:
            seq = np.vstack([np.zeros((N_DAYS - seq.shape[0], N_LAB), np.float32), seq])
        Xlab[k] = seq
        demo = m[0, DEMO]
        icd_flags = (m[:, ICD].max(0) > 0).astype(np.float32)
        med_flags = (m[:, MED].max(0) > 0).astype(np.float32)
        med_int   = m[:, MED].sum(0) / m.shape[0]
        stat.append(np.concatenate([demo, [m.shape[0]], [icd_flags.sum()], icd_flags,
                                    [med_flags.sum()], med_flags, med_int]).astype(np.float32))
    return Xlab, np.vstack(stat).astype(np.float32)

CATEGORICAL = ["gender", "ethnicity"]
Xcat_train = build_flat(train_ids);  Xcat_test = build_flat(test_ids)
Xlab_train, Xstat_train = build_lab_static(train_ids)
Xlab_test,  Xstat_test  = build_lab_static(test_ids)
static_numeric_cols = np.where(Xstat_train.max(0) > 1.5)[0]

print("CatBoost features:", Xcat_train.shape, "->", Xcat_test.shape)
print("GRU lab:", Xlab_train.shape, "-> test", Xlab_test.shape, "| static:", Xstat_train.shape[1])

CatBoost features: (11022, 287) -> (2741, 287)
GRU lab: (11022, 7, 36) -> test (2741, 7, 36) | static: 179


**How to read this:** train and test features built by the same code. Shapes
should match on the feature dimension (only the row count differs). **Remember:**
note the flat-table column count here may differ slightly from the earlier notebook
if dead-column dropping was applied there — CatBoost is robust to that, and here we
build the full set consistently for both train and test.

## 3. Retrain CatBoost on all labeled data, predict test

In [4]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(iterations=500, learning_rate=0.02, depth=7,
                         l2_leaf_reg=20, random_seed=42, verbose=False)
cat.fit(Xcat_train, y_train, cat_features=CATEGORICAL)
p_cat_test = cat.predict_proba(Xcat_test)[:, 1]
print("CatBoost trained on", len(train_ids), "admissions; predicted", len(test_ids), "test rows")
print("test pred range:", round(p_cat_test.min(), 3), "to", round(p_cat_test.max(), 3))

CatBoost trained on 11022 admissions; predicted 2741 test rows
test pred range: 0.024 to 0.992


**How to read this:** the chosen CatBoost, now trained on *all* labeled data
(more than any CV fold saw), predicting test. **Remember:** predictions are
probabilities in [0,1]; the range should look sensible (not all one value).

## 4. Retrain GRU on all labeled data, predict test

We train the v2 GRU on all labeled data. Since there's no validation set here for
early stopping (we're using all data to train), we train for a fixed number of
epochs chosen to match where the CV models typically stopped (~15–20 epochs).

In [5]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score

class GRUStaticReadmission(nn.Module):
    def __init__(self, n_lab, n_static, hidden=64, static_hidden=32, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=n_lab, hidden_size=hidden, batch_first=True)
        self.static_net = nn.Sequential(nn.Linear(n_static, static_hidden), nn.ReLU(), nn.Dropout(dropout))
        self.head = nn.Sequential(nn.Linear(hidden + static_hidden, 32), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, x_lab, x_static):
        _, h = self.gru(x_lab)
        return self.head(torch.cat([h[-1], self.static_net(x_static)], dim=1)).squeeze(-1)

# scale static numerics: fit on ALL labeled data (this is now our full training set)
Xs_tr = Xstat_train.copy(); Xs_te = Xstat_test.copy()
mu = Xs_tr[:, static_numeric_cols].mean(0); sd = Xs_tr[:, static_numeric_cols].std(0) + 1e-6
Xs_tr[:, static_numeric_cols] = (Xs_tr[:, static_numeric_cols] - mu) / sd
Xs_te[:, static_numeric_cols] = (Xs_te[:, static_numeric_cols] - mu) / sd   # test uses TRAIN stats

torch.manual_seed(42)
model = GRUStaticReadmission(N_LAB, Xstat_train.shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
pw = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

dl = DataLoader(TensorDataset(torch.tensor(Xlab_train), torch.tensor(Xs_tr), torch.tensor(y_train)),
                batch_size=128, shuffle=True)
EPOCHS = 18   # fixed; ~ where CV early-stopping landed
for ep in range(EPOCHS):
    model.train()
    for xb_lab, xb_st, yb in dl:
        xb_lab, xb_st, yb = xb_lab.to(device), xb_st.to(device), yb.to(device)
        opt.zero_grad(); loss_fn(model(xb_lab, xb_st), yb).backward(); opt.step()

model.eval()
with torch.no_grad():
    p_gru_test = torch.sigmoid(
        model(torch.tensor(Xlab_test).to(device), torch.tensor(Xs_te).to(device))).cpu().numpy()
print("GRU trained for", EPOCHS, "epochs on all labeled data; predicted", len(test_ids), "test rows")
print("test pred range:", round(float(p_gru_test.min()), 3), "to", round(float(p_gru_test.max()), 3))

GRU trained for 18 epochs on all labeled data; predicted 2741 test rows
test pred range: 0.014 to 1.0


**How to read this:** the GRU trained on all labeled data and predicted test.
**Important caveat:** with no held-out validation here, we train a *fixed* 18
epochs rather than early-stopping. If you want to be safe, hold out a small slice
of labeled data just to pick the epoch count, then retrain on all — but a fixed
count near the CV stopping point is a reasonable, common shortcut. Note test static
features are scaled with **training** statistics (no test leakage).

## 5. Combine by rank average and write the submission

We rank-average the two models' test predictions (scale-safe), then write the
submission file. Rank-averaging on test alone is fine because ranks are computed
within the test set — we're ordering test admissions by combined risk, which is
what AUROC scores.

In [6]:
from scipy.stats import rankdata

r_cat = rankdata(p_cat_test) / len(p_cat_test)
r_gru = rankdata(p_gru_test) / len(p_gru_test)
p_ensemble = (r_cat + r_gru) / 2   # final combined score in [0,1]

submission = pd.DataFrame({"id": test_ids, LABEL_COL: p_ensemble})
out_path = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(out_path, index=False)

print("wrote", out_path, "with", len(submission), "rows")
print(submission.head())
print("\nscore range:", round(p_ensemble.min(), 3), "to", round(p_ensemble.max(), 3))

wrote data\submission.csv with 2741 rows
                  id  readmitted_within_30days
0  19661325_29884966                  0.629880
1  16300198_24781018                  0.180409
2  16706302_24530345                  0.989238
3  13205882_26134830                  0.633528
4  14671276_21169914                  0.828165

score range: 0.004 to 1.0


**How to read this:** the submission file is written — one combined risk score
per test admission. **Remember:** these are the deliverable. Our best estimate of
their AUROC is the OOF ~0.812 (possibly a touch lower on the true test, since the
ensemble weight/choice was tuned on OOF). **Check the grader's expected format**
(column names, whether it wants probabilities or 0/1 labels) and adjust if needed —
some graders want a hard label, in which case threshold `p_ensemble`.

## 6. Done

This completes the project pipeline: EDA → feature engineering → CatBoost model →
feature-selection check → GRU sequence model → ensemble → submission.

**Final model:** rank-average ensemble of CatBoost (flat features) and a
lab-focused GRU, estimated AUROC ~0.812 (OOF), trained on all labeled data for the
submission. See `PROJECT.md` for the full write-up and the honest caveats.